# CHO detector feasibility experiment
Human tracking markers. Evaluation split: validation. No Biohub claims.

In [ ]:
import base64,io,zipfile,hashlib,json,subprocess,sys
from pathlib import Path
import requests
root=Path('/kaggle/working/ctc_project');root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--only-binary=:all:','imagecodecs==2024.12.30'],check=True)
import imagecodecs
assert imagecodecs.lzw_decode(imagecodecs.lzw_encode(b'CTC codec preflight'))==b'CTC codec preflight'
payload='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:z.extractall(root)
protocol=json.loads((root/'frozen_protocol.json').read_text())
for name,expected in protocol['source_hashes'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest()==expected,name
data=root/'data';data.mkdir(exist_ok=True)
archive=data/'Fluo-N3DH-CHO.zip'
response=requests.get(protocol['dataset_url'],stream=True,timeout=(30,120));response.raise_for_status()
size=0;digest=hashlib.sha256()
with archive.open('wb') as f:
    for chunk in response.iter_content(1024*1024):
        size+=len(chunk)
        if size>128*1024*1024:raise RuntimeError('Archive exceeds predeclared 128 MiB limit')
        f.write(chunk);digest.update(chunk)
assert digest.hexdigest()==protocol['dataset_sha256'],'Dataset bytes differ from verified local source'
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert (data/name).resolve().is_relative_to(data.resolve()),name
    z.extractall(data)
scripts=root/'advanced/scripts'
import tifffile
for name in ['01/t000.tif','01_GT/TRA/man_track000.tif']:
    sample=tifffile.imread(data/'Fluo-N3DH-CHO'/name)
    assert sample.ndim==3,(name,sample.shape)
print('LZW raw-image and annotation preflight passed',flush=True)
(root/'comparison').mkdir(exist_ok=True)
subprocess.run([sys.executable,str(scripts/'test_ctc_training.py')],check=True)
subprocess.run([sys.executable,str(scripts/'train_ctc_detector.py'),'--epochs','30','--steps','30'],check=True)
subprocess.run([sys.executable,str(scripts/'compare_ctc_detector.py'),'--split',protocol['evaluation_split']],check=True)
print('CHO DETECTOR TRAINING AND COMPARISON COMPLETED',protocol['evaluation_split'],flush=True)
